# ALEE — Foil ("Negative") Generation

This notebook takes the parallel source texts, generates four kinds of
meaning-altering English **foils** (negatives) with an AMR round-trip + NLI
validation, and writes the three CSVs that feed the embedding pre-computation
notebook (`2--ALEE_PRE-CALCULATE-Embeddings.ipynb`):

| Section | Output | Level | Foil text column |
|---|---|---|---|
| **Flores200** | `datasets/alee_f200.csv`  | sentence            | `foil_<t>_eng_Latn` |
| **MT61**      | `datasets/alee_mt61.csv`  | paragraph           | `foil_<t>_text` |
| **BOUQUET275**| `datasets/alee_bq275.csv` | sentence + paragraph| `foil_<t>_eng_Latn` |

Foil types `<t>`: `polarity_negation`, `role_swap`, `antonym_replacement`, `hypernym_substitution`.

**Input.** Each section loads the parallel originals from the published dataset
[`Psychias/alee_datasets`](https://huggingface.co/datasets/Psychias/alee_datasets)
(any pre-existing `*_negative` columns are dropped and regenerated here). To start
from the raw upstream sources instead, see the commented alternative in each loader.

**Compatibility note.** MT61 keeps Romansh under its original **5-character** `rm_*`
codes (not `roh_*`), because the embeddings notebook discovers languages by a
5-character rule — `roh_puter` would be skipped, `rm_PU` is embedded.


## Setup

Run the install cells once (they target a fresh Colab/VM). Skip if your environment already has the stack.

In [ ]:
# Dependencies (pinned: amrlib 0.8.0 needs transformers < 4.50).
# word2number is required by amrlib's generate model at load time.
# Skip in an environment that already has these installed.
!pip install -q amrlib==0.8.0 "transformers>=4.40,<4.50" word2number nltk penman unidecode smatch pandas tqdm scikit-learn sentence-transformers datasets
!python -m spacy download en_core_web_sm
import amrlib
assert hasattr(amrlib, "setup_spacy_extension"), "amrlib install looks broken"
print("amrlib", amrlib.__version__, "OK")

In [ ]:
# Download the amrlib model weights (AMR-to-text generator + parser) once.
import os, shutil, tarfile, requests
from pathlib import Path
import amrlib

DATA = Path(amrlib.__file__).parent / "data"
MODELS = {
    "model_gtos": ("https://github.com/bjascob/amrlib-models/releases/download/"
                   "model_generate_t5wtense-v0_1_0/model_generate_t5wtense-v0_1_0.tar.gz",
                   "model_generate_t5wtense-v0_1_0"),
    "model_stog": ("https://github.com/bjascob/amrlib-models/releases/download/"
                   "parse_xfm_bart_large-v0_1_0/model_parse_xfm_bart_large-v0_1_0.tar.gz",
                   "model_parse_xfm_bart_large-v0_1_0"),
}
for sub, (url, folder) in MODELS.items():
    dst = DATA / sub
    dst.mkdir(parents=True, exist_ok=True)
    if any(dst.iterdir()):
        print(f"{sub}: already present"); continue
    print(f"Downloading {sub} ...")
    tar = f"/tmp/{sub}.tar.gz"
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(tar, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    tmp = Path(f"/tmp/{sub}_x"); tmp.mkdir(exist_ok=True)
    with tarfile.open(tar, "r:gz") as t:
        t.extractall(tmp, filter="data")
    for item in (tmp / folder).iterdir():
        shutil.move(str(item), str(dst))
    print(f"{sub}: installed")
print("amrlib models ready")

## Models & imports

Loads spaCy + amrlib (AMR generation) and the NLI validator once, for all three sections.

In [ ]:
import os, re, ssl, random, logging
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import nltk
from nltk.tokenize import sent_tokenize
from nltk.corpus import wordnet as wn
from tqdm.auto import tqdm
import amrlib, spacy, penman
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
from huggingface_hub import login

logging.getLogger("penman").setLevel(logging.ERROR)
try:
    ssl._create_default_https_context = ssl._create_unverified_context
except AttributeError:
    pass
for res in ["punkt", "punkt_tab", "wordnet", "averaged_perceptron_tagger"]:
    try:
        nltk.data.find(f"tokenizers/{res}")
    except LookupError:
        nltk.download(res, quiet=True)

tqdm.pandas()
random.seed(42)                      # reproducibility
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(">>> device:", DEVICE)

amrlib.setup_spacy_extension()
NLP = spacy.load("en_core_web_sm")
GTOS = amrlib.load_gtos_model(device=DEVICE)                     # AMR -> text

NLI_NAME = "juliussteen/DeBERTa-v3-FaithAug"
NLI_TOK = AutoTokenizer.from_pretrained(NLI_NAME)
NLI = AutoModelForSequenceClassification.from_pretrained(NLI_NAME).to(DEVICE)
assert NLI.config.id2label[0] == "entailment", "NLI label 0 must be entailment"
print(">>> models loaded")

## Foil operators

WordNet lookups and the four AMR-triple transforms. Pronoun nodes are never targeted.

In [ ]:
PRONOUNS = {"i","you","he","she","it","we","they","me","him","her","us","them"}

def _best_synsets(word):
    prefix = word.split("-")[0] if "-" in word else word
    best = []
    for pos in (wn.NOUN, wn.VERB, wn.ADJ, wn.ADV):
        s = wn.synsets(prefix, pos=pos)
        if len(s) > len(best):
            best = s
    return best

def get_hypernyms(word):
    out = set()
    for syn in _best_synsets(word):
        if syn.hypernyms():
            for h in syn.hypernyms():
                out.update(h.lemma_names())
        else:
            out.update(syn.lemma_names())
    return list(out)

def get_antonyms(word):
    out = set()
    for syn in _best_synsets(word):
        for lemma in syn.lemmas():
            for ant in lemma.antonyms():
                out.add(ant.name())
    return list(out)

def _leaf_nodes_and_edges(triples):
    g = defaultdict(list)
    for s, r, o in triples:
        if r != ":instance":
            g[s].append((r, o)); g[o].append((r, s))
    return {(n, g[n][0]) for n in g if len(g[n]) == 1}

def role_swap(triples):
    """Swap two leaf arguments (who-did-what-to-whom altered)."""
    leaves = list(_leaf_nodes_and_edges(triples))
    if len(leaves) < 2:
        return triples
    (n1, e1), (n2, e2) = random.sample(leaves, 2)
    new = []
    for s, r, o in triples:
        if   (s, r, o) == (n1, e1[0], e1[1]): new.append((n1, e2[0], e2[1]))
        elif (s, r, o) == (e1[1], e1[0], n1): new.append((e2[1], e2[0], n1))
        elif (s, r, o) == (n2, e2[0], e2[1]): new.append((n2, e1[0], e1[1]))
        elif (s, r, o) == (e2[1], e2[0], n2): new.append((e1[1], e1[0], n2))
        else: new.append((s, r, o))
    return new

def polarity_negation(triples):
    """Negate a proposition by adding a :polarity - edge."""
    nodes = list({s for s, r, o in triples if o not in PRONOUNS})
    if not nodes:
        return triples
    return triples + [(random.choice(nodes), ":polarity", "-")]

def antonym_replacement(triples):
    """Replace a concept with a WordNet antonym."""
    cand = [(s, r, o) for s, r, o in triples if r == ":instance" and o not in PRONOUNS]
    while cand:
        s, r, o = random.choice(cand); cand.remove((s, r, o))
        prefix, suffix = (o.split("-", 1) + [None])[:2] if "-" in o else (o, None)
        ants = get_antonyms(prefix)
        if ants:
            new_o = random.choice(ants) + ("-" + suffix if suffix is not None else "")
            return [(s, r, new_o) if (s, r, o) == (a, b, c) else (a, b, c) for a, b, c in triples]
    return triples

def hypernym_substitution(triples):
    """Over-generalise a concept with a WordNet hypernym."""
    cand = [(s, r, o) for s, r, o in triples if r == ":instance" and o not in PRONOUNS]
    if not cand:
        return triples
    s, r, o = random.choice(cand)
    prefix, suffix = (o.split("-", 1) + [None])[:2] if "-" in o else (o, None)
    hyps = get_hypernyms(prefix)
    if hyps:
        new_o = random.choice(hyps) + ("-" + suffix if suffix is not None else "")
        return [(s, r, new_o) if (s, r, o) == (a, b, c) else (a, b, c) for a, b, c in triples]
    return triples

# transform name -> operator + min input length
FOIL_PIPELINES = {
    "polarity_negation":     {"augment": polarity_negation,     "min_len": 25},
    "role_swap":             {"augment": role_swap,             "min_len": 25},
    "antonym_replacement":   {"augment": antonym_replacement,   "min_len": 25},
    "hypernym_substitution": {"augment": hypernym_substitution, "min_len": 25},
}

## AMR generation & NLI validation

A foil is a **success** only if it is *not* bidirectionally entailing and neither direction exceeds 0.8 entailment probability — i.e. the meaning genuinely changed.

In [ ]:
def _to_amr(text):
    return NLP(text)._.to_amr()

def _generate(mod_triples):
    enc = penman.encode(penman.Graph(mod_triples))
    try:
        gen = GTOS.generate([enc], disable_progress=True)
    except TypeError:
        gen = GTOS.generate([enc])
    if not gen:
        return None
    return gen[0][0] if isinstance(gen[0], list) else gen[0]

def _post_process(original, foil):
    if not foil:
        return None
    if original and original[0].isupper():
        foil = foil[0].upper() + foil[1:]
    return re.sub(r"\s+([.,;!?])", r"\1", foil)

def _clean(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def _nli(premise, hypothesis):
    inp = NLI_TOK(premise, hypothesis, return_tensors="pt", truncation=True).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(NLI(**inp).logits, dim=-1).cpu().numpy()[0]
    return (int(np.argmax(probs)) == 0), float(probs[0])   # (is_entailment, entail_prob)

def validate_foil(original, foil, threshold=0.8):
    f_ok, f_p = _nli(original, foil)
    b_ok, b_p = _nli(foil, original)
    info = {"forward": {"prob": f_p, "is_entailment": f_ok},
            "backward": {"prob": b_p, "is_entailment": b_ok}}
    if f_ok and b_ok:
        return False, "failed_bidirectional_entailment", info
    if f_p > threshold or b_p > threshold:
        return False, "failed_high_probability", info
    return True, "success", info

def _score(info):
    f, b = info["forward"]["is_entailment"], info["backward"]["is_entailment"]
    return 2.0 if (f and b) else (1.0 if (f or b) else 0.0)

## Row processors

`process_sentence_row` (best-of-all scoring) for FLORES/BOUQuET-sentence; `process_paragraph_row` (greedy first-valid, NLI on the full paragraph) for MT61/BOUQuET-paragraph.

In [ ]:
def process_sentence_row(text):
    best = {k: {"text": text, "status": "no_change", "probs": None, "priority": 0}
            for k in FOIL_PIPELINES}
    if pd.isna(text) or not isinstance(text, str) or len(text) < 25:
        return _fmt_sentence(best)
    try:
        graphs = _to_amr(text)
        if not graphs:
            return _fmt_sentence(best)
        gstr = graphs[0]
        for name, pipe in FOIL_PIPELINES.items():
            if len(text) < pipe["min_len"]:
                continue
            g = penman.decode(gstr)
            mod = pipe["augment"](g.triples)
            if mod == g.triples:
                continue
            raw = _generate(mod)
            if not raw:
                continue
            foil = _post_process(text, raw)
            if not foil or _clean(text) == _clean(foil):
                continue
            ok, status, info = validate_foil(text, foil)
            prio = 2 if ok else 1
            cur = _score(info)
            ex = best[name]
            ex_score = _score(ex["probs"]) if ex["probs"] else float("inf")
            if prio > ex["priority"] or (prio == ex["priority"] and cur < ex_score):
                best[name] = {"text": foil, "status": ("success" if ok else status),
                              "probs": info, "priority": prio}
    except Exception:
        pass
    return _fmt_sentence(best)

def _fmt_sentence(best):
    out = {}
    for k, d in best.items():
        p = d["probs"]
        out[f"foil_{k}_eng_Latn"] = d["text"]
        out[f"foil_{k}_status"] = d["status"]
        out[f"foil_{k}_entailment_fwd_prob"] = p["forward"]["prob"] if p else None
        out[f"foil_{k}_entailment_bwd_prob"] = p["backward"]["prob"] if p else None
        out[f"foil_{k}_is_entailment_fwd"] = p["forward"]["is_entailment"] if p else None
        out[f"foil_{k}_is_entailment_bwd"] = p["backward"]["is_entailment"] if p else None
    return pd.Series(out)

def _process_one_paragraph(text, pipe):
    res = {"foil_text": text, "status": "no_change", "orig_sent": None, "foil_sent": None,
           "fwd": None, "bwd": None, "efwd": None, "ebwd": None}
    if pd.isna(text) or not isinstance(text, str) or len(text) < pipe["min_len"]:
        return res
    sents = sent_tokenize(text)
    targets = sents if len(sents) > 1 else [text]
    for sent in targets:
        if len(sents) > 1 and len(sent) < 20:
            continue
        try:
            graphs = _to_amr(sent)
            if not graphs:
                continue
            g = penman.decode(graphs[0])
            mod = pipe["augment"](g.triples)
            if mod == g.triples:
                continue
            raw = _generate(mod)
            if not raw:
                continue
            foil_sent = _post_process(sent, raw)
            if not foil_sent or _clean(sent) == _clean(foil_sent):
                continue
            foil_par = text.replace(sent, foil_sent, 1) if len(sents) > 1 else foil_sent
            ok, status, info = validate_foil(text, foil_par)
            if ok or res["status"] == "no_change":
                res.update({"foil_text": foil_par, "status": status,
                            "orig_sent": sent, "foil_sent": foil_sent,
                            "fwd": info["forward"]["prob"], "bwd": info["backward"]["prob"],
                            "efwd": info["forward"]["is_entailment"],
                            "ebwd": info["backward"]["is_entailment"]})
            if ok:
                break                       # greedy: stop at first success
        except Exception:
            pass
    return res

def process_paragraph_row(text):
    row = {}
    for name, pipe in FOIL_PIPELINES.items():
        r = _process_one_paragraph(text, pipe)
        row[f"foil_{name}_eng_Latn"] = r["foil_text"]
        row[f"foil_{name}_status"] = r["status"]
        row[f"foil_{name}_original_sentence"] = r["orig_sent"]
        row[f"foil_{name}_foil_sentence"] = r["foil_sent"]
        row[f"foil_{name}_entailment_fwd_prob"] = r["fwd"]
        row[f"foil_{name}_entailment_bwd_prob"] = r["bwd"]
        row[f"foil_{name}_is_entailment_fwd"] = r["efwd"]
        row[f"foil_{name}_is_entailment_bwd"] = r["ebwd"]
    return row

## Config & helpers

Output folder and small helpers shared by the three sections.

In [ ]:
OUTPUT_DIR = "datasets"                 # where the embeddings notebook reads its CSVs
os.makedirs(OUTPUT_DIR, exist_ok=True)
HF_DATASET = "Psychias/alee_datasets"

# Romansh: published roh_* -> original 5-char rm_* (so the embeddings notebook,
# which discovers languages by a 5-char rule, still finds them).
ROH_TO_RM = {"roh_rumgr": "rm_RG", "roh_sursilv": "rm_SV", "roh_sutsilv": "rm_ST",
             "roh_surmiran": "rm_SM", "roh_puter": "rm_PU", "roh_vallader": "rm_VA"}

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])

def drop_negatives(df):
    """Remove any pre-existing published foil columns; we regenerate them here."""
    return df.drop(columns=[c for c in df.columns if c.endswith("_negative")])

# The published dataset drops the `sentence_` prefix on language columns; the foil
# generation + the embeddings notebook expect it, so we re-add it on load.
META_FLORES = {"id", "URL", "domain", "topic", "has_image", "has_hyperlink", "SIB_CATEGORY"}
META_BOUQUET = {"id", "uniq_id", "domain", "register", "tags", "level", "split",
                "par_id", "par_comment", "orig_text", "newline_next"}

def add_sentence_prefix(df, meta):
    """Re-add `sentence_` to language columns (everything that is not metadata)."""
    return df.rename(columns={c: f"sentence_{c}" for c in df.columns if c not in meta})

def overall_success_mask(df):
    m = pd.Series(False, index=df.index)
    for t in FOIL_PIPELINES:
        m |= (df[f"foil_{t}_status"] == "success")
    return m

## 1 · Flores200 → `datasets/alee_f200.csv`

Sentence-level. Foils are scored best-of-all per transform; we keep the rows with at
least one successful foil.

In [ ]:
# --- load originals (published config has no `sentence_` prefix; re-add it) ---
df = load_dataset(HF_DATASET, "alee_f200", split="test").to_pandas()
df = add_sentence_prefix(drop_negatives(df), META_FLORES)
# Alternative (raw source): load_dataset("Muennighoff/flores200","all",split="devtest",
#                                        trust_remote_code=True).to_pandas()
if "id" not in df.columns:
    df.insert(0, "id", range(1, len(df) + 1))
print("flores rows:", len(df))

# --- generate foils on the English column ---
foils = df["sentence_eng_Latn"].progress_apply(process_sentence_row)
flores = pd.concat([df, foils], axis=1)

# --- reorder (id, foil texts, statuses, probs, then languages) & keep rows with a success ---
text_cols   = [c for c in flores.columns if c.startswith("foil_") and c.endswith("_eng_Latn")]
status_cols = [c for c in flores.columns if c.startswith("foil_") and c.endswith("_status")]
prob_cols   = [c for c in flores.columns if c.startswith("foil_") and "_entailment_" in c]
other       = [c for c in flores.columns if c not in ["id"] + text_cols + status_cols + prob_cols]
flores = flores[["id"] + text_cols + status_cols + prob_cols + other]
flores = flores[overall_success_mask(flores)]

out = os.path.join(OUTPUT_DIR, "alee_f200.csv")
flores.to_csv(out, index=False, encoding="utf-16")
print(f"saved {len(flores)} rows -> {out}")

## 2 · MT61 (WMT24++ incl. Romansh) → `datasets/alee_mt61.csv`

Paragraph-level. Romansh is written under 5-char `rm_*` codes. The WMT24 foil text
column is `foil_<t>_text` (the embeddings notebook expects that suffix).

In [ ]:
# --- load originals (English + 55 targets + 6 Romansh), remap roh_* -> rm_* ---
df = load_dataset(HF_DATASET, "alee_mt61", split="test").to_pandas()
df = drop_negatives(df).rename(columns=ROH_TO_RM)
# Alternative (raw source): build en_EN + <lang> from google/wmt24pp and merge
#   ZurichNLP/wmt24pp-rm on segment_id (see amr_foil_pipeline.py: prep_wmt24).
print("mt61 rows:", len(df), "| languages:",
      len([c for c in df.columns if len(c) == 5 and c[2] == "_"]))

# --- generate foils on en_EN (paragraph pipeline) ---
records = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="MT61"):
    r = process_paragraph_row(row["en_EN"])
    records.append({k.replace("_eng_Latn", "_text"): v for k, v in r.items()})  # _text suffix
mt61 = pd.concat([df.reset_index(drop=True), pd.DataFrame(records)], axis=1)

out = os.path.join(OUTPUT_DIR, "alee_mt61.csv")
mt61.to_csv(out, index=False, encoding="utf-16")
print(f"saved {len(mt61)} rows -> {out}  (Romansh: "
      f"{[c for c in mt61.columns if c.startswith('rm_')]})")

## 3 · BOUQUET275 → `datasets/alee_bq275.csv`

Mixed: `sentence_level` rows use the sentence pipeline, `paragraph_level` rows use the
paragraph pipeline. Kept rows have at least one successful foil; the `level` column
is preserved.

In [ ]:
# --- load originals for both levels and tag them ---
# single merged config, distinguished by the `level` column; re-add `sentence_`
df = drop_negatives(load_dataset(HF_DATASET, "alee_bq275", split="test").to_pandas())
df = add_sentence_prefix(df, META_BOUQUET)
print("bouquet rows:", len(df), "| levels:", df["level"].value_counts().to_dict())

# --- sentence rows -> sentence pipeline; paragraph rows -> paragraph pipeline ---
sent = df[df["level"] == "sentence_level"].copy()
para = df[df["level"] == "paragraph_level"].copy()

sent_foils = sent["sentence_eng_Latn"].progress_apply(process_sentence_row)
sent_final = pd.concat([sent, sent_foils], axis=1)

para_records = [process_paragraph_row(t) for t in
                tqdm(para["sentence_eng_Latn"], desc="BOUQUET paragraphs")]
para_final = pd.concat([para, pd.DataFrame(para_records, index=para.index)], axis=1)

# align columns across the two levels, then combine
for c in para_final.columns:
    if c not in sent_final.columns: sent_final[c] = None
for c in sent_final.columns:
    if c not in para_final.columns: para_final[c] = None
bouquet = pd.concat([sent_final, para_final], ignore_index=True)
if "id" not in bouquet.columns:
    bouquet.insert(0, "id", range(1, len(bouquet) + 1))
bouquet = bouquet[overall_success_mask(bouquet)]

out = os.path.join(OUTPUT_DIR, "alee_bq275.csv")
bouquet.to_csv(out, index=False, encoding="utf-16")
print(f"saved {len(bouquet)} rows -> {out}")